# Lab 13 — 스팸 분류기 완성하기

**확률통계 · Week 13 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. **2주차에 시작한 스팸 필터를 완성**한다 (확률 0 문제 해결).
2. Laplace smoothing이 **Beta prior의 posterior 평균**임을 코드로 확인한다.
3. **순차 갱신** — 데이터를 하나씩 봐도 결과가 같음을 확인한다.

⏱ **예상 소요 시간: 35분**

> 2주차 랩에서 우리는 벽에 부딪혔다. 학습 데이터에 없는 단어 하나가
> 전체 판정을 무너뜨렸다. 오늘 그 문제를 **prior로** 해결한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)

mails = [
    ("spam", "free money click now win"),
    ("spam", "win free prize click link"),
    ("spam", "urgent free offer click here"),
    ("spam", "you win money now urgent"),
    ("spam", "click link free lottery win"),
    ("spam", "cheap offer free shipping the deal"),
    ("ham",  "meeting tomorrow at ten"),
    ("ham",  "please review the report"),
    ("ham",  "lunch at noon today"),
    ("ham",  "the lecture note is uploaded"),
    ("ham",  "can you send me the free file"),
    ("ham",  "homework deadline is friday"),
    ("ham",  "thanks for the quick review"),
    ("ham",  "see you at the meeting"),
    ("ham",  "please click the attached file"),
]

spam_mails = [t.split() for lab, t in mails if lab == "spam"]
ham_mails = [t.split() for lab, t in mails if lab == "ham"]
p_spam = len(spam_mails) / len(mails)
p_ham = 1 - p_spam

print(f"스팸 {len(spam_mails)}통 / 정상 {len(ham_mails)}통")

## Part 1. 2주차의 문제 재현

먼저 **prior 없이** (= MLE) 계산하면 어떻게 되는지 다시 본다.

In [ ]:
def word_prob_mle(word, mails_list):
    count = sum(1 for words in mails_list if word in words)
    return count / len(mails_list)


def classify_mle(text):
    score_spam, score_ham = p_spam, p_ham
    for w in text.split():
        score_spam *= word_prob_mle(w, spam_mails)
        score_ham *= word_prob_mle(w, ham_mails)
    if score_spam + score_ham == 0:
        return None
    return score_spam / (score_spam + score_ham)


for t in ["free click", "free money pizza", "free money meeting"]:
    print(f"{t:>22} -> {classify_mle(t)}")

`pizza` 하나 때문에 **판정 불가(None)** 가 된다. 2주차의 그 문제다.

## Part 2. Beta prior 넣기

$$\hat p = \frac{k + \alpha}{n + \alpha + \beta}
\qquad (\alpha = \beta = 1 \text{ 이면 } \frac{k+1}{n+2})$$

### 실습 1 — smoothing된 확률

In [ ]:
ALPHA, BETA = 1.0, 1.0

def word_prob_bayes(word, mails_list, alpha=ALPHA, beta=BETA):
    k = sum(1 for words in mails_list if word in words)
    n = len(mails_list)
    # TODO 1: posterior 평균 (k + alpha) / (n + alpha + beta) 를 돌려주세요
    return 0.0


for w in ["free", "pizza", "meeting"]:
    print(f"{w:>9}  MLE spam {word_prob_mle(w, spam_mails):.4f}"
          f" -> Bayes {word_prob_bayes(w, spam_mails):.4f}")

> `pizza` 의 확률이 **0이 아니라 작은 양수**가 되었다.
> "본 적 없다"와 "절대 없다"는 다르다는 것을 수식으로 표현한 것이다.

### 실습 2 — 분류기 다시 만들기

In [ ]:
def classify_bayes(text, alpha=ALPHA, beta=BETA):
    log_spam = np.log(p_spam)
    log_ham = np.log(p_ham)
    for w in text.split():
        # TODO 2: 각 단어의 log 확률을 더해 나가세요
        #         힌트: log_spam += np.log(word_prob_bayes(w, spam_mails, alpha, beta))
        pass
    m = max(log_spam, log_ham)
    s, h = np.exp(log_spam - m), np.exp(log_ham - m)
    return s / (s + h)


for t in ["free click", "free money pizza", "free money meeting",
          "meeting tomorrow", "pizza pizza pizza"]:
    print(f"{t:>22} -> P[spam] = {classify_bayes(t):.4f}")

🎉 **더 이상 `None` 이 나오지 않는다.**

- `free money pizza` — 모르는 단어가 있어도 아는 단어로 판정한다
- `pizza pizza pizza` — 아는 단어가 하나도 없는데 **0.63** 이 나왔다.
  prior $\mathbb{P}[\text{spam}] = 0.4$ 보다 오히려 높다. **왜일까?**

> 🤔 **생각해보기** — 스팸은 6통, 정상은 9통이다.
> 못 본 단어의 확률은 스팸 쪽이 $(0+1)/(6+2) = 0.125$, 정상 쪽이 $(0+1)/(9+2) = 0.091$ 이다.
> **표본이 적은 클래스에서 smoothing 효과가 더 크게 작용**해 그쪽으로 기운다.
> 실무에서 클래스 불균형이 있을 때 조심해야 하는 지점이다.

## Part 3. prior의 세기를 바꿔보면

$\alpha$ 를 키우면 "모든 단어가 반반이다"라는 믿음이 강해진다.

### 실습 3

In [ ]:
for a in [0.1, 1.0, 5.0, 50.0]:
    # TODO 3: alpha=beta=a 로 두 문장을 분류해보세요
    #         힌트: classify_bayes(t, alpha=a, beta=a)
    probs = [0.5, 0.5]
    print(f"alpha=beta={a:>5}   'free click' {probs[0]:.4f}   "
          f"'meeting tomorrow' {probs[1]:.4f}")

🤔 **$\alpha$ 가 커질수록 판정이 0.5(중립)에 가까워진다.**

prior가 "모든 단어는 스팸/정상에 반반씩 나온다"고 강하게 주장하기 때문에
데이터 15통으로는 그 믿음을 이기지 못하는 것이다.

> **prior가 세면 데이터가 묻힌다.** 그래서 약한 prior부터 시작하는 것이 원칙이다.

## Part 4. 순차 갱신 — 계산 순서는 상관없다

동전 데이터로 확인한다. **한 번에 다 보기** vs **하나씩 보기**.

### 실습 4

In [ ]:
data = (rng.random(14) < 0.6).astype(int)
print("데이터:", data, f"(앞면 {data.sum()}개)")

a_batch = 1 + data.sum()
b_batch = 1 + (len(data) - data.sum())

a_seq, b_seq = 1.0, 1.0
for x in data:
    # TODO 4: 관측값 x 에 따라 a_seq 또는 b_seq 를 1 늘리세요
    #         힌트: a_seq += x  그리고  b_seq += (1 - x)
    pass

print(f"\n한 번에 : Beta({a_batch}, {b_batch})")
print(f"하나씩  : Beta({a_seq:.0f}, {b_seq:.0f})")
print("같은가?", (a_batch, b_batch) == (a_seq, b_seq))

### 실습 5 — posterior가 좁아지는 과정 그리기

In [ ]:
theta = np.linspace(0, 1, 300)
plt.figure(figsize=(7.5, 4.2))

a, b = 1.0, 1.0
for i, x in enumerate(data):
    a += x
    b += (1 - x)
    if (i + 1) in [1, 3, 7, 14]:
        plt.plot(theta, stats.beta.pdf(theta, a, b), lw=2,
                 label=f"n={i + 1}  Beta({a:.0f},{b:.0f})")

plt.axvline(0.6, color="black", ls=":", lw=2, label="true 0.6")
plt.xlabel("theta")
plt.ylabel("posterior density")
plt.title("Posterior gets narrower as data accumulates")
plt.legend()
plt.show()

print(f"최종 posterior 평균 {a / (a + b):.4f}")
# TODO 5: 95% credible interval 을 구하세요
#         힌트: stats.beta.interval(0.95, a, b)
print("95% credible interval", None)

---

## 마무리 — 자가 점검

- [ ] Laplace smoothing이 Beta prior의 posterior 평균임을 확인했다
- [ ] 처음 보는 단어가 있어도 분류기가 작동하는 것을 확인했다
- [ ] prior가 세면 데이터가 묻힌다는 것을 관찰했다
- [ ] 순차 갱신과 일괄 갱신의 결과가 같음을 확인했다
- [ ] credible interval을 계산할 수 있다

**2주차에 만난 문제가 오늘 어떻게 풀렸는지 한 문장으로.**

> (여기에 작성)

### 📌 미니 프로젝트 2에서 베이지안 추정을 쓴다면 **prior와 그 근거를 반드시 보고서에 쓸 것**